In [17]:
import pandas as pd
df = pd.read_csv('../data/raw/June_8_data_metro_closest_stations.csv')
df.head()

,Unnamed: 0,size_sq_ft,propertyType,bedrooms,latitude,longitude,localityName,suburbName,cityName,price,companyName,closest_mtero_station_km,AP_dist_km,Aiims_dist_km,NDRLW_dist_km
0,0,400,Independent Floor,1,28.641010,77.284386,Swasthya Vihar,Delhi East,Delhi,9000,Dream Homez,0.577495,21.741188,11.119239,6.227231
1,1,1050,Apartment,2,28.594969,77.298668,mayur vihar phase 1,Delhi East,Delhi,20000,Rupak Properties Stock,0.417142,21.401856,9.419061,9.217502
2,2,2250,Independent Floor,2,28.641806,77.293922,Swasthya Vihar,Delhi East,Delhi,28000,Aashiyana Real Estate,0.125136,22.620365,11.829486,7.159184
3,3,1350,Independent Floor,2,28.644363,77.293228,Krishna Nagar,Delhi East,Delhi,28000,Shivam Real Estate,0.371709,22.681201,11.982708,7.097348
4,4,450,Apartment,2,28.594736,77.311150,New Ashok Nagar,Delhi East,Delhi,12500,Shree Properties,1.087760,22.592810,10.571573,10.263271


In [18]:
print(f"Shape: {df.shape}\n")
print("--- Info ---")
df.info()
print(f"\n--- Describe ---\n{df.describe()}")

Shape: (17890, 15)

--- Info ---
<class 'pandas.DataFrame'>
RangeIndex: 17890 entries, 0 to 17889
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                17890 non-null  int64  
 1   size_sq_ft                17890 non-null  int64  
 2   propertyType              17890 non-null  str    
 3   bedrooms                  17890 non-null  int64  
 4   latitude                  17890 non-null  float64
 5   longitude                 17890 non-null  float64
 6   localityName              17890 non-null  str    
 7   suburbName                17890 non-null  str    
 8   cityName                  17890 non-null  str    
 9   price                     17890 non-null  int64  
 10  companyName               17890 non-null  str    
 11  closest_mtero_station_km  17890 non-null  float64
 12  AP_dist_km                17890 non-null  float64
 13  Aiims_dist_km             17890 non-nul

In [19]:
df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
17885    False
17886    False
17887    False
17888    False
17889    False
Length: 17890, dtype: bool

In [20]:
df.isnull().sum().sum()

np.int64(0)

In [21]:
print(df['price'].quantile([0.01, 0.02, 0.05, 0.95, 0.98, 0.99]))
print(df['size_sq_ft'].quantile([0.01, 0.02, 0.05, 0.95, 0.98, 0.99]))

0.01      5500.00
0.02      6500.00
0.05      8000.00
0.95     85000.00
0.98    150000.00
0.99    222030.96
Name: price, dtype: float64
0.01     210.0
0.02     260.0
0.05     350.0
0.95    2650.0
0.98    3500.0
0.99    4500.0
Name: size_sq_ft, dtype: float64


In [22]:
n_before = len(df)

price_low, price_high = df['price'].quantile([0.01, 0.99])
size_low, size_high = df['size_sq_ft'].quantile([0.01, 0.99])

price_mask = df['price'].between(price_low, price_high)
size_mask = df['size_sq_ft'].between(size_low, size_high)

print(f"Rows dropped by price outlier filter: {(~price_mask).sum()}")
print(f"Rows dropped by size outlier filter: {(~size_mask).sum()}")
print(f"Rows dropped by both: {(~price_mask & ~size_mask).sum()}")

df_clean = df[price_mask & size_mask].copy()
n_after = len(df_clean)

print(f"Total: {n_before} -> {n_after} rows ({n_before - n_after} dropped, {100*(n_before-n_after)/n_before:.1f}%)")

Rows dropped by price outlier filter: 290
Rows dropped by size outlier filter: 303
Rows dropped by both: 91
Total: 17890 -> 17388 rows (502 dropped, 2.8%)


# Outlier Removal Summary:
To remove extreme values, the bottom 1% and top 1% percentiles were trimmed for both price and size_sq_ft.

Price Outliers Dropped: 290 rows

Size Outliers Dropped: 303 rows

Overlapping Outliers (Both): 91 rows

Dataset Impact

Initial Rows: 17,890

Cleaned Rows: 17,388

Total Removed: 502 rows (2.8% of total data)

In [23]:
localities = df_clean['localityName'].unique()
print(len(localities), "unique raw locality strings")

# Look for casing/whitespace issues
import re
sample = sorted(localities)[:40]
for s in sample:
    print(repr(s))

750 unique raw locality strings
'10 Sector Dwarka'
'100 Feet Road'
'40 Feet Road'
'45 Sector 21 Road'
'48 Sector 22 Road'
'52 Sector Road'
'60 Feet Road'
'72 Sector 23 Road'
'75 Noida Road'
'75 Sector 22 Road'
'76 Noida Road'
'94 B Block Road'
'944 B Block Road'
'A 2 Block'
'A 3 Block'
'A 6 Block'
'A1 Block Paschim Vihar Delhi'
'A4 Block Paschim Vihar'
'A5 Block'
'AC Block Shalimar Bagh'
'AD Block Pitampura'
'AGCR Enclave'
'AKSHARDHAM'
'APB Garhi'
'Abul Fazal Enclave Jamia Nagar'
'Abul Fazal Enclave Part 2 Jamia Nagar'
'Acharya Niketan'
'Adarsh Nagar'
'Adchini'
'Ajmeri Gate'
'Alaknanda'
'Alaknanda DDA Flats Road'
'Alaknanda Gangotri Enclave'
'Alipur Village'
'Amar colony'
'Anand Kunj'
'Anand Nagar Laxmi Nagar'
'Anand Niketan'
'Anand Parbat'
'Anand Vihar'


# Raw Locality Inspection
A inspection of the first sorted unique localityName values reveals key patterns in how neighborhoods are formatted:

Inconsistent Naming Formats: Sector names appear in multiple style variations (e.g., '10 Sector Dwarka' vs 'Sector 1 Dwarka').

Micro-Level Details: Listings frequently include hyper-local identifiers such as specific road names ('100 Feet Road'), block codes ('A 2 Block'), or building markers ('AD Block Pitampura').

Noise and Spelling Variations: Instances of uppercase noise ('AKSHARDHAM'), shorthand variations ('APB Garhi'), and combined parent-child names ('Anand Nagar Laxmi Nagar') are present.

In [24]:
lower_localities = df_clean['localityName'].str.strip().str.lower()
print(lower_localities.nunique(), "unique after lowercasing + stripping whitespace")

750 unique after lowercasing + stripping whitespace


# Text Standardization Check
Running lowercasing and whitespace trimming on localityName yields 750 unique values, which matches the exact count from the previous raw frequency check.

Key Takeaways

No Hidden Duplicates: There are no casing inconsistencies (e.g., "Kalkaji" vs "kalkaji") or trailing/leading space issues artificially inflating the unique neighborhood count.

Original Casing Preserved: Since lowercasing does not reduce the number of unique categories, you can safely keep the title-cased names for better presentation without missing any duplicate merges.

In [25]:
locality_counts = df_clean['localityName'].value_counts()
print(locality_counts.describe())
print()
print("Localities with only 1 listing:", (locality_counts == 1).sum())
print("Localities with <5 listings:", (locality_counts < 5).sum())
print("Localities with <10 listings:", (locality_counts < 10).sum())
print()
print("% of total rows in localities with <5 listings:",
      100 * df_clean[df_clean['localityName'].isin(locality_counts[locality_counts<5].index)].shape[0] / len(df_clean))

count     750.000000
mean       23.184000
std        86.291621
min         1.000000
25%         1.000000
50%         2.000000
75%         9.000000
max      1441.000000
Name: count, dtype: float64

Localities with only 1 listing: 273
Localities with <5 listings: 482
Localities with <10 listings: 563

% of total rows in localities with <5 listings: 4.738900391074304


# Locality Distribution & Sparsity Analysis
An evaluation of listing counts across unique neighborhoods (localityName) reveals high skewness, with a large number of long-tail, low-density localities.

Summary Statistics

Total Unique Localities: 750

Mean Listings per Locality: ~23.2

Median Listings per Locality: 2.0 (50% of localities have 2 or fewer listings)

Max Listings in a Single Locality: 1,441

Low-Density Localities Breakdown

1 Listing: 273 localities

< 5 Listings: 482 localities (64.3% of all unique localities)

< 10 Listings: 563 localities (75.1% of all unique localities)

Dataset Coverage Impact

Listings in sparse localities (<5 entries): 4.74% of total dataset rows (~824 listings).

Key Takeaway: While 64.3% of unique neighborhood names are extremely sparse (<5 listings), they only account for ~4.7% of the entire dataset. Flagging or grouping these sparse localities prevents model overfitting without sacrificing significant data volume.

In [26]:
# Tier 1: coarse, reliable grouping (already clean, ~10-12 values)
df_clean['localityGroup'] = df_clean['suburbName'].str.strip()

# Tier 2: keep raw fine-grained locality, but flag sparse ones
locality_counts = df_clean['localityName'].value_counts()
df_clean['locality_listing_count'] = df_clean['localityName'].map(locality_counts)
df_clean['is_sparse_locality'] = df_clean['locality_listing_count'] < 5

print(df_clean[['localityName','localityGroup','locality_listing_count','is_sparse_locality']].sample(10))

          localityName  localityGroup  locality_listing_count  \
7015     Paschim Vihar     West Delhi                     946   
14409         Om Vihar  Delhi Central                      17   
17837   Rajinder Nagar     Delhi West                     260   
3228   East of Kailash    Delhi South                     142   
8464   Gujranwala Town    North Delhi                      66   
580        Preet Vihar     Delhi East                     239   
5741   East of Kailash    Delhi South                     142   
12535      Patel Nagar  Delhi Central                    1441   
2956         Hauz Khas    Delhi South                      48   
17683   Rajinder Nagar     Delhi West                     260   

       is_sparse_locality  
7015                False  
14409               False  
17837               False  
3228                False  
8464                False  
580                 False  
5741                False  
12535               False  
2956                False  
1768

# Locality Hierarchy & Sparsity Feature Engineering
To handle granular locations effectively without creating noisy categories, a two-tiered geographic feature structure was created:

Tier 1 (localityGroup): Clean, high-level regional clusters (~10–12 unique values like Dwarka, Delhi South, Delhi East) derived from suburbName.

Tier 2 (localityName + Sparsity Flags): Preserves specific neighborhood details while tracking sample density.

locality_listing_count: Total count of listings in that specific neighborhood.

is_sparse_locality: Boolean flag marking rare localities with fewer than 5 listings (< 5).

In [28]:
from pathlib import Path
import sqlite3
import pandas as pd

# ------------------------------------------------------------------
# Dynamic Path Resolution
# ------------------------------------------------------------------
# Jupyter notebooks do not define __file__. Locate the project root
# from the current working directory instead.
cwd = Path.cwd().resolve()
data_filename = "June_8_data_metro_closest_stations.csv"

candidates = [cwd, *cwd.parents]
BASE_DIR = next(
    (
        path
        for path in candidates
        if (path / "data" / "raw" / data_filename).exists()
    ),
    cwd.parent if cwd.name == "notebooks" else cwd,
)
DATA_PATH = BASE_DIR / "data" / "raw" / "June_8_data_metro_closest_stations.csv"
DB_PATH = BASE_DIR / "db" / "rent_fairness.db"

# Ensure db output directory exists
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 1. Load Data & Clean
# ------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)

# Outlier filtering
price_low, price_high = df["price"].quantile([0.01, 0.99])
size_low, size_high = df["size_sq_ft"].quantile([0.01, 0.99])

price_mask = df["price"].between(price_low, price_high)
size_mask = df["size_sq_ft"].between(size_low, size_high)

df_clean = df[price_mask & size_mask].copy()

# Feature engineering
df_clean["localityGroup"] = df_clean["suburbName"].str.strip()
locality_counts = df_clean["localityName"].value_counts()
df_clean["locality_listing_count"] = df_clean["localityName"].map(locality_counts)
df_clean["is_sparse_locality"] = df_clean["locality_listing_count"] < 5

# ------------------------------------------------------------------
# 2. Build 'localities' Table
# ------------------------------------------------------------------
localities_df = (
    df_clean.groupby("localityName")
    .agg(
        locality_group=("localityGroup", "first"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean"),
        listing_count=("locality_listing_count", "first"),
        is_sparse=("is_sparse_locality", "first"),
    )
    .reset_index()
    .rename(columns={"localityName": "locality_name"})
)

localities_df.insert(0, "locality_id", range(1, len(localities_df) + 1))

# ------------------------------------------------------------------
# 3. Build 'listings' Table
# ------------------------------------------------------------------
locality_map = dict(zip(localities_df["locality_name"], localities_df["locality_id"]))
listings_df = df_clean.copy()
listings_df["locality_id"] = listings_df["localityName"].map(locality_map)

if "id" not in listings_df.columns:
    listings_df["listing_id"] = range(1, len(listings_df) + 1)

listings_cols = {
    "listing_id": "listing_id",
    "locality_id": "locality_id",
    "size_sq_ft": "size_sq_ft",
    "property_type": "property_type",
    "bedrooms": "bedrooms",
    "price": "price",
    "company_name": "company_name",
    "closest_metro_km": "closest_metro_km",
    "airport_dist_km": "airport_dist_km",
    "aiims_dist_km": "aiims_dist_km",
    "ndrlw_dist_km": "ndrlw_dist_km",
}

# Filter to available columns safely
existing_cols = [c for c in listings_cols.keys() if c in listings_df.columns]
listings_df = listings_df[existing_cols].rename(columns=listings_cols)

# ------------------------------------------------------------------
# 4. Build Empty 'amenities' Table
# ------------------------------------------------------------------
amenities_df = pd.DataFrame(columns=["amenity_id", "listing_id", "amenity_name"])

# ------------------------------------------------------------------
# 5. Export to SQLite
# ------------------------------------------------------------------
conn = sqlite3.connect(DB_PATH)

localities_df.to_sql("localities", conn, if_exists="replace", index=False)
listings_df.to_sql("listings", conn, if_exists="replace", index=False)
amenities_df.to_sql("amenities", conn, if_exists="replace", index=False)

with conn:
    conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_locality_id ON localities(locality_id);")
    conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_listing_id ON listings(listing_id);")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_listing_locality ON listings(locality_id);")

conn.close()
print(f"Successfully generated relational database at {DB_PATH}")

Successfully generated relational database at D:\desktop\rent-fairness-ncr\db\rent_fairness.db
